# Train the spiking decoder (SNN) alongside the RNN
Runs after `rnn_step1_makeTFRecords.ipynb` (or with the TFRecords from `derived.tar.gz`) and can run alongside `rnn_step2_trainBaselineRNN.ipynb`; `rnn_step3_baselineRNNInference.ipynb` then evaluates both side by side.

Two stages, as in the velocity SNN's pipeline (see `SNNDecoder/README.md`):
1. **pretrain** on all 19 training sessions with hard reset
2. **fine-tune** from it with soft reset (the reset has no weights, so the pretrained weights load unchanged)

Needs `SNNDecoder` installed (`pip install -e SNNDecoder`); the export step needs TensorFlow, training needs PyTorch + sinabs. On a Slurm cluster, `SNNDecoder/scripts/run_snn_pretrain.sbatch` and `run_snn_finetune.sbatch` run the same two stages.

In [ ]:
#to run on your machine, change baseDir
import os
baseDir = '/users/$USER/scratch/speechBCI_dryad'
os.environ['BASE_DIR'] = baseDir
os.environ['SNN_CONFIG'] = os.path.abspath('../SNNDecoder/snnDecoder/configs/speech_pooled.yaml')
os.makedirs(baseDir+'/derived/snns', exist_ok=True)

In [ ]:
%%bash
#export the RNN's TFRecords to .npz, so both decoders use identical features and labels (TensorFlow env)
python3 exportTFRecordsToNpz.py --tfRecordDir $BASE_DIR/derived/tfRecords --outDir $BASE_DIR/derived/snnData

In [ ]:
%%bash
#stage 1: pretrain on every training session, hard reset
python3 -m snnDecoder.train \
    --config $SNN_CONFIG \
    --data-dir $BASE_DIR/derived/snnData \
    --checkpoint-dir $BASE_DIR/derived/snns/pretrain_hard \
    --reset-type hard

In [ ]:
%%bash
#stage 2: fine-tune from the pretrained weights, soft reset (fresh optimizer)
python3 -m snnDecoder.train \
    --config $SNN_CONFIG \
    --data-dir $BASE_DIR/derived/snnData \
    --checkpoint-dir $BASE_DIR/derived/snns/finetune_soft \
    --init-weights-from $BASE_DIR/derived/snns/pretrain_hard/best_model_weights.pth \
    --reset-type soft

In [ ]:
#training curves (CTC loss, test phoneme error rate, learning rate) of both stages
from IPython.display import Image, display
for stage, reset in [('pretrain_hard', 'hard'), ('finetune_soft', 'soft')]:
    display(Image(filename=baseDir + '/derived/snns/' + stage + '/training_curves_speech_iaf_' + reset + '.png'))